<a href="https://colab.research.google.com/github/sowmya-prabha/SNN_on_NMnist/blob/main/SNN_on_NMnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [45]:
!kaggle datasets download -d khoahongg/n-mnist

Dataset URL: https://www.kaggle.com/datasets/khoahongg/n-mnist
License(s): unknown
n-mnist.zip: Skipping, found more recently modified local copy (use --force to force download)


In [46]:
!unzip -o n-mnist.zip

Streaming output truncated to the last 5000 lines.
  inflating: Train/Train/9/09730.bin  
  inflating: Train/Train/9/09732.bin  
  inflating: Train/Train/9/09752.bin  
  inflating: Train/Train/9/09753.bin  
  inflating: Train/Train/9/09755.bin  
  inflating: Train/Train/9/09759.bin  
  inflating: Train/Train/9/09772.bin  
  inflating: Train/Train/9/09785.bin  
  inflating: Train/Train/9/09792.bin  
  inflating: Train/Train/9/09811.bin  
  inflating: Train/Train/9/09818.bin  
  inflating: Train/Train/9/09826.bin  
  inflating: Train/Train/9/09828.bin  
  inflating: Train/Train/9/09836.bin  
  inflating: Train/Train/9/09839.bin  
  inflating: Train/Train/9/09847.bin  
  inflating: Train/Train/9/09871.bin  
  inflating: Train/Train/9/09877.bin  
  inflating: Train/Train/9/09890.bin  
  inflating: Train/Train/9/09894.bin  
  inflating: Train/Train/9/09897.bin  
  inflating: Train/Train/9/09934.bin  
  inflating: Train/Train/9/09939.bin  
  inflating: Train/Train/9/09942.bin  
  inflating: 

In [47]:
import os
os.listdir()

['.config',
 'Test',
 'Train',
 'n-mnist.zip',
 '.ipynb_checkpoints',
 'kaggle.json',
 'sample_data']

In [ ]:
import numpy as np

def read_nmnist(Train):
    with open(Train, 'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
    data = data.reshape(-1, 5)

    x = data[:, 0]
    y = data[:, 1]
    polarity = (data[:, 2] >> 7) & 1
    timestamp = ((data[:, 2] & 127) << 16) | \
                (data[:, 3] << 8) | \
                data[:, 4]
    return x, y, polarity, timestamp

In [50]:
import os
print(os.listdir("/content/Train/Train/0"))
files_in_dir = os.listdir("/content/Train/Train/0")
if files_in_dir:
    sample_file_path = os.path.join("/content/Train/Train/0", files_in_dir[0])
    x, y, p, t = read_nmnist(sample_file_path)

    print("Number of events:", len(x))
    print("X:", x[:10])
    print("Y:", y[:10])
    print("Polarity:", p[:10])
    print("Timestamp:", t[:10])
else:
    print("The directory '/content/Train/Train/0' is empty.")

['53815.bin', '12636.bin', '41229.bin', '54550.bin', '44860.bin', '21404.bin', '07131.bin', '53340.bin', '15735.bin', '25920.bin', '30587.bin', '01490.bin', '19925.bin', '10222.bin', '18869.bin', '36352.bin', '03563.bin', '47423.bin', '30161.bin', '53510.bin', '15029.bin', '06046.bin', '51500.bin', '30059.bin', '26776.bin', '58727.bin', '30735.bin', '54405.bin', '47167.bin', '35328.bin', '04202.bin', '46413.bin', '47406.bin', '46563.bin', '15696.bin', '50821.bin', '23754.bin', '53074.bin', '19739.bin', '30701.bin', '57721.bin', '44697.bin', '21603.bin', '27138.bin', '53862.bin', '44449.bin', '51492.bin', '18744.bin', '01502.bin', '52186.bin', '32494.bin', '18882.bin', '55278.bin', '06682.bin', '50241.bin', '20549.bin', '57761.bin', '26819.bin', '20293.bin', '01388.bin', '41466.bin', '49060.bin', '40123.bin', '05463.bin', '53536.bin', '06620.bin', '29542.bin', '51506.bin', '42171.bin', '03480.bin', '44868.bin', '28273.bin', '44364.bin', '43046.bin', '49200.bin', '06338.bin', '31467.bin'

In [53]:
!pip install spikingjelly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 5.9 MB/s eta 0:00:00


In [54]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class NMNISTDataset(Dataset):

    def __init__(self, root, time_steps=10):
        self.root = root
        self.time_steps = time_steps
        self.samples = []
        for label in range(10):

            class_path = os.path.join(root, str(label))

            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):

                if file.endswith(".bin"):
                    file_path = os.path.join(class_path, file)
                    self.samples.append((file_path, label))

        print("Total samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def read_events(self, filename):

        with open(filename, "rb") as f:
            data = np.frombuffer(f.read(), dtype=np.uint8)

        # Each event = 5 bytes
        data = data.reshape(-1, 5)

        x = data[:, 0]
        y = data[:, 1]

        polarity = (data[:, 2] >> 7) & 1

        timestamp = (
            ((data[:, 2] & 127) << 16)
            | (data[:, 3] << 8)
            | data[:, 4]
        )

        return x, y, polarity, timestamp

    def __getitem__(self, index):

        filename, label = self.samples[index]

        x, y, p, t = self.read_events(filename)

        # Create time bins
        frames = np.zeros(
            (self.time_steps, 2, 34, 34),
            dtype=np.float32
        )

        # Normalize timestamp
        t_min = t.min()
        t_max = t.max()

        if t_max > t_min:
            time_index = (
                (t - t_min) /
                (t_max - t_min) *
                (self.time_steps - 1)
            ).astype(int)
        else:
            time_index = np.zeros_like(t)

        # Put events into frames
        for i in range(len(x)):

            ti = time_index[i]

            if x[i] < 34 and y[i] < 34:
                frames[ti, p[i], y[i], x[i]] = 1

        frames = torch.tensor(frames)

        return frames, torch.tensor(label)

In [55]:
train_dataset = NMNISTDataset(
    "/content/Train/Train",
    time_steps=10
)

test_dataset = NMNISTDataset(
    "/content/Test/Test",
    time_steps=10
)

Total samples: 60000
Total samples: 10000


In [56]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [57]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 10, 2, 34, 34])
torch.Size([32])


In [60]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.2 MB/s eta 0:00:00


In [61]:
from spikingjelly.activation_based import neuron
from spikingjelly.activation_based import functional

In [62]:
import torch.nn as nn
import snntorch as snn

In [63]:
class SNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(
            2, 16,
            kernel_size=3,
            padding=1
        )

        self.lif1 = neuron.IFNode()

        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(
            16, 32,
            kernel_size=3,
            padding=1
        )

        self.lif2 = neuron.IFNode()

        self.pool2 = nn.MaxPool2d(2)

        self.fc = nn.Linear(
            32 * 8 * 8,
            10
        )

        self.lif3 = neuron.IFNode()

    def forward(self, x):

        # x:
        # [batch, time, channel, height, width]

        outputs = []

        for t in range(x.shape[1]):

            current = x[:, t]

            current = self.conv1(current)
            current = self.lif1(current)
            current = self.pool1(current)

            current = self.conv2(current)
            current = self.lif2(current)
            current = self.pool2(current)

            current = current.flatten(1)

            current = self.fc(current)
            current = self.lif3(current)

            outputs.append(current)

        return torch.stack(outputs)

In [64]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SNN().to(device)

print(device)
print(model)

cpu
SNN(
  (conv1): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (lif1): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=False, step_mode=s, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (lif2): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=False, step_mode=s, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=2048, out_features=10, bias=True)
  (lif3): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=False, step_mode=s, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
)


In [65]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [70]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Reset neuron states
        functional.reset_net(model)

        outputs = model(images)

        # outputs:
        # [time, batch, classes]

        # Sum spikes over time
        outputs = outputs.sum(dim=0)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {total_loss:.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

Exception ignored in atexit callback <function _exit_function at 0x7b0720604cc0>:
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/util.py", line 431, in _exit_function
    _run_finalizers()
  File "/usr/lib/python3.13/multiprocessing/util.py", line 371, in _run_finalizers
    finalizer()
  File "/usr/lib/python3.13/multiprocessing/util.py", line 295, in __call__
    res = self._callback(*self._args, **self._kwargs)
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 217, in _finalize_join
    thread.join()
  File "/usr/lib/python3.13/threading.py", line 1095, in join
    self._handle.join(timeout)
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        functional.reset_net(model)

        outputs = model(images)

        outputs = outputs.sum(dim=0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)